# MoBE Interactive Walkthrough

This notebook is a walkthrough for MoBE. It uses BiomedCLIP inference, local modality-expert checkpoints, and the same no-gradient MoBE routing/adaptation logic used by the repository scripts. The dashboards report model outputs from the selected image or dataset.

**How To Use This Notebook**

You do not need to read or edit the code to follow the demo. Run each numbered cell from top to bottom. When a cell shows controls, use the dropdowns, sliders, and buttons. The heavy steps are button-driven so a naive user can move at their own pace without being surprised by a long model load.

**What You Will See**

| Cell | Purpose |
|---|---|
| Cell 1 | Setup imports, paths, device, widgets, and shared helpers. |
| Cell 2 | Pick one bundled sample image and show its label set. |
| Cell 3 | Load BiomedCLIP and local MoBE expert checkpoints. |
| Cell 4 | Run a single-image BiomedCLIP-vs-MoBE dashboard. |
| Cell 5 | Run a live one-dataset BiomedCLIP-vs-MoBE comparison. |
| Cell 6 | Summarize the workflow and how it maps to the repo scripts. |

**Important Notes**

- The sample-image dashboard does not load a dataset; it uses the six images in `assets/samples/`.
- The model cells require BiomedCLIP to be available to `open_clip` and expert checkpoints to exist under `experts/` or the configured fallback path.
- The live dataset panel runs one dataset at a time. To compare with `.py` scripts, keep `Use full dataset`, `Use config lambda`, and `Config route` enabled.


## Cell 1 — Setup

Run this first. It prepares imports, paths, widgets, random seeds, and the compute device.

In [1]:
import html
import math
import os
import random
import time
import traceback
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd()
ASSET_DIR = PROJECT_ROOT / "assets"
SAMPLES_DIR = ASSET_DIR / "samples"
CONFIG_ROOT = PROJECT_ROOT / "configs"
DATA_ROOTS = [
    PROJECT_ROOT / "datasets_all",
    Path("/home/razaimam/Documents/Projects/TTW/datasets_all"),
]
EXPERT_SEARCH_DIRS = [
    PROJECT_ROOT / "experts",
    Path("/home/razaimam/Documents/Projects/TTW/experts"),
]
MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
MODALITIES = ["Angiogram", "CT", "MRI", "Ultrasound", "Xray"]
DEFAULT_LIVE_DATASET = "chestmnist_224"


def seed_everything(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}. Widgets available: {HAS_WIDGETS}")


Running on cuda. Widgets available: True


## Cell 2 — Select A Bundled Sample

Choose one sample image from the notebook UI. The image, label space, prompt templates, and MoBE config source are selected together. This cell does not run a model yet.

In [ ]:
COVID19_LABELS = [
    "covid_lungs",
    "lung_opacity_lungs",
    "normal_lungs",
    "viral_pneumonia_lungs",
]

BTMRI_LABELS = [
    "glioma_tumor",
    "meningioma_tumor",
    "normal_brain",
    "pituitary_tumor",
]

DERMAMNIST_LABELS = [
    "actinic_keratosis",
    "basal_cell_carcinoma",
    "benign_keratosis",
    "dermatofibroma",
    "melanocytic_nevus",
    "melanoma",
    "vascular_lesion",
]

IMAGE_PRESETS = {
    "covid19_covid_lungs": {
        "path": SAMPLES_DIR / "COVID19_covid_lungs.png",
        "dataset": "COVID-19",
        "display_name": "COVID-19 | covid_lungs",
        "labels": COVID19_LABELS,
        "templates": [
            "a chest x-ray showing {}.",
            "a radiology image of {}.",
            "a medical lung image consistent with {}.",
        ],
        "ground_truth": "covid_lungs",
        "config_dataset": "hardbench_covid19",
    },
    "covid19_normal_lungs": {
        "path": SAMPLES_DIR / "COVID19_normal_lungs.png",
        "dataset": "COVID-19",
        "display_name": "COVID-19 | normal_lungs",
        "labels": COVID19_LABELS,
        "templates": [
            "a chest x-ray showing {}.",
            "a radiology image of {}.",
            "a medical lung image consistent with {}.",
        ],
        "ground_truth": "normal_lungs",
        "config_dataset": "hardbench_covid19",
    },
    "btmri_glioma_tumor": {
        "path": SAMPLES_DIR / "image_BTMRI_glioma_tumor.jpg",
        "dataset": "BTMRI",
        "display_name": "BTMRI | glioma_tumor",
        "labels": BTMRI_LABELS,
        "templates": [
            "a brain MRI showing {}.",
            "a medical image of {}.",
            "an MRI scan consistent with {}.",
        ],
        "ground_truth": "glioma_tumor",
        "config_dataset": "hardbench_btmri",
    },
    "btmri_normal_brain": {
        "path": SAMPLES_DIR / "image_BTMRI_normal_brain.jpg",
        "dataset": "BTMRI",
        "display_name": "BTMRI | normal_brain",
        "labels": BTMRI_LABELS,
        "templates": [
            "a brain MRI showing {}.",
            "a medical image of {}.",
            "an MRI scan consistent with {}.",
        ],
        "ground_truth": "normal_brain",
        "config_dataset": "hardbench_btmri",
    },
    "dermamnist_actinic_keratosis": {
        "path": SAMPLES_DIR / "dermamnist_actinic_keratosis.png",
        "dataset": "DermaMNIST",
        "display_name": "DermaMNIST | actinic_keratosis",
        "labels": DERMAMNIST_LABELS,
        "templates": [
            "a dermatoscopic image of {}.",
            "a skin lesion image consistent with {}.",
            "a medical dermatology image showing {}.",
        ],
        "ground_truth": "actinic_keratosis",
        "config_dataset": "dermamnist",
    },
    "dermamnist_melanoma": {
        "path": SAMPLES_DIR / "dermamnist_melanoma.png",
        "dataset": "DermaMNIST",
        "display_name": "DermaMNIST | melanoma",
        "labels": DERMAMNIST_LABELS,
        "templates": [
            "a dermatoscopic image of {}.",
            "a skin lesion image consistent with {}.",
            "a medical dermatology image showing {}.",
        ],
        "ground_truth": "melanoma",
        "config_dataset": "dermamnist",
    },
}

DEFAULT_SAMPLE_ID = "dermamnist_melanoma"
SAMPLE_OPTIONS = [(preset["display_name"], key) for key, preset in IMAGE_PRESETS.items()]
SELECTED_SAMPLE_ID = DEFAULT_SAMPLE_ID
SELECTED_SAMPLE = None
SELECTED_IMAGE = None


def prompt_label(classname):
    return str(classname).replace("_", " ")


def load_sample(sample_id=DEFAULT_SAMPLE_ID):
    global SELECTED_SAMPLE_ID, SELECTED_SAMPLE, SELECTED_IMAGE
    SELECTED_SAMPLE_ID = sample_id
    SELECTED_SAMPLE = IMAGE_PRESETS[sample_id]
    image_path = Path(SELECTED_SAMPLE["path"])
    if not image_path.exists():
        raise FileNotFoundError(f"Missing sample image: {image_path}")
    SELECTED_IMAGE = Image.open(image_path).convert("RGB")
    return SELECTED_SAMPLE, SELECTED_IMAGE


def show_sample(sample_id=DEFAULT_SAMPLE_ID):
    sample, image = load_sample(sample_id)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
    axes[0].imshow(image, cmap="gray")
    axes[0].axis("off")
    axes[0].set_title(sample["display_name"])
    label_text = "\n".join(f"{idx}: {label}" for idx, label in enumerate(sample["labels"]))
    axes[1].axis("off")
    axes[1].text(
        0.0,
        1.0,
        f"Ground truth: {sample['ground_truth']}\nConfig: {sample['config_dataset']}\n\nLabels:\n{label_text}",
        va="top",
        family="monospace",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()


if HAS_WIDGETS:
    sample_picker = widgets.Dropdown(
        options=SAMPLE_OPTIONS,
        value=DEFAULT_SAMPLE_ID,
        description="Sample",
        layout=widgets.Layout(width="520px"),
    )
    sample_output = widgets.Output()

    def _on_sample_change(change=None):
        with sample_output:
            clear_output(wait=True)
            show_sample(sample_picker.value)

    sample_picker.observe(_on_sample_change, names="value")
    display(widgets.VBox([widgets.HTML("<b>Cell 2: choose a sample</b>"), sample_picker, sample_output]))
    _on_sample_change()
else:
    show_sample(DEFAULT_SAMPLE_ID)


## Cell 3 — Load BiomedCLIP And Expert Checkpoints

Click **Load models** once. The following cells reuse the loaded BiomedCLIP model and expert checkpoints.

In [3]:
MODEL_CONTEXT = {"ready": False}


def find_expert_checkpoint(modality):
    filename = f"expert_{modality}_0.pt"
    for experts_dir in EXPERT_SEARCH_DIRS:
        ckpt_path = Path(experts_dir) / filename
        if ckpt_path.exists():
            return ckpt_path
    return None


def load_model_context(force=False, output=None):
    if MODEL_CONTEXT.get("ready") and not force:
        return MODEL_CONTEXT

    context = output if output is not None else nullcontext()
    with context:
        clear_output(wait=True)
        print("Loading BiomedCLIP and modality experts...")

    import open_clip
    from mobe import load_expert_mlp

    model_name = "ViT-B-16" if MODEL_NAME == "ViT-B/16" else MODEL_NAME
    pretrained = None if model_name.startswith("hf-hub:") else "openai"

    base_model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    base_model = base_model.to(device).eval()
    tokenizer = open_clip.get_tokenizer(model_name)

    loaded_modalities = []
    expert_ckpts = []
    for modality in MODALITIES:
        ckpt_path = find_expert_checkpoint(modality)
        if ckpt_path is not None:
            loaded_modalities.append(modality)
            expert_ckpts.append(ckpt_path)

    if not loaded_modalities:
        searched = "\n".join(str(path) for path in EXPERT_SEARCH_DIRS)
        raise FileNotFoundError(f"No expert checkpoints found. Searched:\n{searched}")

    expert_models = []
    load_lines = []
    for modality, ckpt_path in zip(loaded_modalities, expert_ckpts):
        expert_model = open_clip.create_model(model_name, pretrained=pretrained).to(device).eval()
        loaded, missing, unexpected = load_expert_mlp(expert_model, str(ckpt_path))
        expert_models.append(expert_model)
        load_lines.append(
            f"{modality}: {ckpt_path} | loaded_keys={loaded}, missing={len(missing)}, unexpected={len(unexpected)}"
        )

    MODEL_CONTEXT.clear()
    MODEL_CONTEXT.update({
        "ready": True,
        "model_name": model_name,
        "pretrained": pretrained,
        "base_model": base_model,
        "preprocess": preprocess,
        "tokenizer": tokenizer,
        "modalities": loaded_modalities,
        "expert_models": expert_models,
        "expert_ckpts": expert_ckpts,
    })

    with context:
        clear_output(wait=True)
        print("Loaded models successfully.")
        print("Device:", device)
        print("Experts:")
        for line in load_lines:
            print("-", line)

    return MODEL_CONTEXT


if HAS_WIDGETS:
    load_button = widgets.Button(description="Load models", button_style="primary", icon="download")
    reload_button = widgets.Button(description="Reload", icon="refresh")
    load_output = widgets.Output()

    def _load_clicked(_):
        load_button.disabled = True
        reload_button.disabled = True
        try:
            load_model_context(force=False, output=load_output)
        except Exception:
            with load_output:
                clear_output(wait=True)
                traceback.print_exc()
        finally:
            load_button.disabled = False
            reload_button.disabled = False

    def _reload_clicked(_):
        load_button.disabled = True
        reload_button.disabled = True
        try:
            load_model_context(force=True, output=load_output)
        except Exception:
            with load_output:
                clear_output(wait=True)
                traceback.print_exc()
        finally:
            load_button.disabled = False
            reload_button.disabled = False

    load_button.on_click(_load_clicked)
    reload_button.on_click(_reload_clicked)
    display(widgets.VBox([
        widgets.HTML("<b>Cell 3: load model state</b>"),
        widgets.HBox([load_button, reload_button]),
        load_output,
    ]))
else:
    print("Run load_model_context() to load BiomedCLIP and local experts.")


## Cell 4 — Single-Image BiomedCLIP vs MoBE Dashboard

This cell forwards the selected sample through BiomedCLIP and the loaded modality experts. It visualizes BiomedCLIP confidence, MoBE confidence, expert routing scores, and the final selected experts.

In [ ]:
def _cfg_value(cfg, *names, default=None):
    for name in names:
        if name in cfg:
            return cfg[name]
    if default is not None:
        return default
    raise KeyError(f"Missing config key; expected one of {names}")


def _cfg_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "y"}
    return bool(value)


def load_mobe_cfg(dataset_name):
    from utils import get_config_file

    cfg = get_config_file(str(CONFIG_ROOT), dataset_name)
    raw = cfg.get("mobe") or cfg.get("MOBE")
    if raw is None:
        raise KeyError(f"Config for {dataset_name} must contain a mobe/MOBE section.")
    return {
        "use_mobe": _cfg_bool(_cfg_value(raw, "use_mobe", "use_MOBE")),
        "route_with_mobe": _cfg_bool(_cfg_value(raw, "route_with_mobe", "route_with_MOBE")),
        "thr1": float(_cfg_value(raw, "thr1", "MOBE_thr1")),
        "thr2": float(_cfg_value(raw, "thr2", "MOBE_thr2")),
        "c1": int(_cfg_value(raw, "c1", "MOBE_c1")),
        "c2": int(_cfg_value(raw, "c2", "MOBE_c2")),
        "temp": float(_cfg_value(raw, "temp", "MOBE_temp")),
        "lambda": float(_cfg_value(raw, "lambda", "MOBE_lambda")),
    }


def dynamic_k_route(scores, lower_is_better=True, topk=4, gap_thr=0.07):
    scores = torch.as_tensor(scores, device=device)
    sorted_scores, sorted_indices = torch.sort(scores, descending=not lower_is_better)
    best_score = float(sorted_scores[0])
    selected = [int(sorted_indices[0])]
    for rank in range(1, min(int(topk), len(scores))):
        score = float(sorted_scores[rank])
        gap = (score - best_score) if lower_is_better else (best_score - score)
        if gap < float(gap_thr):
            selected.append(int(sorted_indices[rank]))
        else:
            break
    if len(selected) == 1 and len(scores) > 1:
        selected.append(int(sorted_indices[1]))
    return selected, sorted_indices.detach().cpu().tolist(), sorted_scores.detach().cpu().tolist()


def mixture_weights(selected, entropies, strategy="hard", tau=0.07):
    weights = torch.zeros(len(entropies), device=device)
    if strategy == "hard":
        for idx in selected:
            weights[idx] = 1.0 / len(selected)
    elif strategy == "softmax":
        local = torch.softmax((-entropies[selected]) / float(tau), dim=0)
        weights[selected] = local
    else:
        local = 1.0 / (torch.exp(entropies[selected]) + 1e-12)
        weights[selected] = local / local.sum()
    return weights


def mobe_init_from_text_weights(clip_weights, num_experts, init_c1, init_c2):
    mu0 = clip_weights.t().to(device)
    mu = mu0.unsqueeze(0).repeat(num_experts, 1, 1).contiguous()
    pi = torch.eye(mu0.size(0), device=device).unsqueeze(0).repeat(num_experts, 1, 1)
    c1 = torch.full((num_experts, mu0.size(0)), float(init_c1), device=device)
    c2 = torch.full((num_experts, mu0.size(0)), float(init_c2), device=device)
    return mu, pi, c1, c2


def mobe_forward_and_update_sample(feature, expert_idx, mu, pi, c1, c2, thr1, thr2, temp):
    cluster_logits = temp * (feature @ mu[expert_idx].t())
    cluster_probs = cluster_logits.softmax(dim=-1)
    mobe_probs = cluster_probs @ pi[expert_idx]
    confidence, prediction = mobe_probs.max(dim=-1)
    confidence_value = float(confidence.mean().item())
    predicted_class = int(prediction[0].item())

    updated_mu = False
    updated_pi = False
    if confidence_value > thr1:
        mu[expert_idx, predicted_class] = (
            c1[expert_idx, predicted_class] * mu[expert_idx, predicted_class] + feature[0]
        ) / (c1[expert_idx, predicted_class] + 1.0)
        mu[expert_idx, predicted_class] = F.normalize(mu[expert_idx, predicted_class], dim=0)
        c1[expert_idx, predicted_class] += 1.0
        updated_mu = True

    if confidence_value > thr2:
        pi[expert_idx, predicted_class] = (
            c2[expert_idx, predicted_class] * pi[expert_idx, predicted_class] + mobe_probs[0]
        ) / (c2[expert_idx, predicted_class] + 1.0)
        c2[expert_idx, predicted_class] += 1.0
        updated_pi = True

    return mobe_probs, confidence_value, updated_mu, updated_pi


def run_single_image_comparison(sample_id=None, route_override=None, lambda_override=None, gap_thr=0.07, topk=4, tau=0.07, mix_strategy="hard"):
    from utils import clip_classifier, get_clip_logits

    if sample_id is None:
        sample_id = sample_picker.value if HAS_WIDGETS and "sample_picker" in globals() else SELECTED_SAMPLE_ID
    sample, image = load_sample(sample_id)
    context = load_model_context(force=False)
    mobe_cfg = load_mobe_cfg(sample["config_dataset"])
    if route_override is not None:
        mobe_cfg["route_with_mobe"] = bool(route_override)
    if lambda_override is not None:
        mobe_cfg["lambda"] = float(lambda_override)

    classnames = sample["labels"]
    templates = sample["templates"]
    clip_weights = clip_classifier(classnames, templates, context["base_model"], context["tokenizer"]).to(device)
    image_tensor = context["preprocess"](image).unsqueeze(0).to(device)

    with torch.no_grad():
        _, biomed_logits, _, _, _ = get_clip_logits(image_tensor, context["base_model"], clip_weights)
        biomed_probs = biomed_logits.softmax(dim=-1)
        biomed_conf, biomed_pred = biomed_probs.max(dim=-1)

        mu, pi, c1, c2 = mobe_init_from_text_weights(
            clip_weights,
            len(context["expert_models"]),
            init_c1=mobe_cfg["c1"],
            init_c2=mobe_cfg["c2"],
        )

        expert_feature_list = []
        expert_logit_list = []
        expert_prob_list = []
        entropies = torch.empty(len(context["expert_models"]), device=device)
        mobe_confidences = torch.empty(len(context["expert_models"]), device=device)
        update_flags = []

        for expert_idx, expert_model in enumerate(context["expert_models"]):
            features, logits, _, _, _ = get_clip_logits(image_tensor, expert_model, clip_weights)
            features = F.normalize(features, dim=-1)
            probs = logits.softmax(dim=-1)
            entropy = -(probs * (probs + 1e-12).log()).sum(dim=-1).mean()
            _, confidence, did_mu, did_pi = mobe_forward_and_update_sample(
                features,
                expert_idx,
                mu,
                pi,
                c1,
                c2,
                thr1=mobe_cfg["thr1"],
                thr2=mobe_cfg["thr2"],
                temp=mobe_cfg["temp"],
            )
            expert_feature_list.append(features)
            expert_logit_list.append(logits)
            expert_prob_list.append(probs.squeeze(0))
            entropies[expert_idx] = entropy
            mobe_confidences[expert_idx] = confidence
            update_flags.append((did_mu, did_pi))

        if mobe_cfg["use_mobe"] and mobe_cfg["route_with_mobe"]:
            selected, ranking, _ = dynamic_k_route(
                mobe_confidences,
                lower_is_better=False,
                topk=min(int(topk), len(context["expert_models"])),
                gap_thr=gap_thr,
            )
            route_scores = mobe_confidences
            route_name = "MoBE confidence"
        else:
            selected, ranking, _ = dynamic_k_route(
                entropies,
                lower_is_better=True,
                topk=min(int(topk), len(context["expert_models"])),
                gap_thr=gap_thr,
            )
            route_scores = entropies
            route_name = "Entropy"

        weights = mixture_weights(selected, entropies, strategy=mix_strategy, tau=tau)
        final_logits = torch.zeros_like(expert_logit_list[0])
        for expert_idx in selected:
            final_logits += weights[expert_idx] * expert_logit_list[expert_idx]

        if mobe_cfg["use_mobe"] and mobe_cfg["lambda"] > 0:
            mobe_mix_probs = 0.0
            for expert_idx in selected:
                cluster_logits = mobe_cfg["temp"] * (expert_feature_list[expert_idx] @ mu[expert_idx].t())
                cluster_probs = cluster_logits.softmax(dim=-1)
                adapted_probs = cluster_probs @ pi[expert_idx]
                mobe_mix_probs = mobe_mix_probs + weights[expert_idx] * adapted_probs
            final_logits = (1.0 - mobe_cfg["lambda"]) * final_logits + mobe_cfg["lambda"] * torch.log(mobe_mix_probs + 1e-12)

        mobe_probs = final_logits.softmax(dim=-1)
        mobe_conf, mobe_pred = mobe_probs.max(dim=-1)

    return {
        "sample": sample,
        "image": image,
        "classnames": classnames,
        "mobe_cfg": mobe_cfg,
        "modalities": context["modalities"],
        "biomed_probs": biomed_probs.squeeze(0).detach().cpu().numpy(),
        "biomed_pred": int(biomed_pred.item()),
        "biomed_conf": float(biomed_conf.item()),
        "mobe_probs": mobe_probs.squeeze(0).detach().cpu().numpy(),
        "mobe_pred": int(mobe_pred.item()),
        "mobe_conf": float(mobe_conf.item()),
        "expert_probs": torch.stack(expert_prob_list).detach().cpu().numpy(),
        "entropies": entropies.detach().cpu().numpy(),
        "mobe_confidences": mobe_confidences.detach().cpu().numpy(),
        "route_scores": route_scores.detach().cpu().numpy(),
        "route_name": route_name,
        "selected": selected,
        "ranking": ranking,
        "weights": weights.detach().cpu().numpy(),
        "update_flags": update_flags,
    }


def plot_single_image_result(result):
    classnames = result["classnames"]
    modalities = result["modalities"]
    selected = set(result["selected"])
    fig = plt.figure(figsize=(16, 9.5))
    grid = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.05])
    ax_img = fig.add_subplot(grid[0, 0])
    ax_bio = fig.add_subplot(grid[0, 1])
    ax_mobe = fig.add_subplot(grid[0, 2])
    ax_route = fig.add_subplot(grid[1, 0])
    ax_heat = fig.add_subplot(grid[1, 1])
    ax_weights = fig.add_subplot(grid[1, 2])

    ax_img.imshow(result["image"], cmap="gray")
    ax_img.axis("off")
    ax_img.set_title(result["sample"]["display_name"])

    ax_bio.barh(classnames, result["biomed_probs"], color="#2f6f9f")
    ax_bio.set_xlim(0, 1)
    ax_bio.set_title(f"BiomedCLIP: {classnames[result['biomed_pred']]} ({100 * result['biomed_conf']:.1f}%)")
    ax_bio.set_xlabel("probability")

    ax_mobe.barh(classnames, result["mobe_probs"], color="#476a2a")
    ax_mobe.set_xlim(0, 1)
    ax_mobe.set_title(f"MoBE: {classnames[result['mobe_pred']]} ({100 * result['mobe_conf']:.1f}%)")
    ax_mobe.set_xlabel("probability")

    route_colors = ["#476a2a" if idx in selected else "#b8b8b8" for idx in range(len(modalities))]
    ax_route.bar(modalities, result["route_scores"], color=route_colors)
    ax_route.set_title(f"Routing score: {result['route_name']}")
    ax_route.tick_params(axis="x", rotation=30)
    ax_route.set_ylabel("score")

    heat = ax_heat.imshow(result["expert_probs"], aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax_heat.set_yticks(range(len(modalities)))
    ax_heat.set_yticklabels(modalities)
    ax_heat.set_xticks(range(len(classnames)))
    ax_heat.set_xticklabels([str(i) for i in range(len(classnames))])
    ax_heat.set_title("Expert class probabilities")
    ax_heat.set_xlabel("class id")
    fig.colorbar(heat, ax=ax_heat, fraction=0.046, pad=0.04)

    ax_weights.bar(modalities, result["weights"], color=["#725e9c" if w > 0 else "#d7d7d7" for w in result["weights"]])
    ax_weights.set_ylim(0, 1)
    ax_weights.set_title("Final mixture weights")
    ax_weights.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.show()

    print("Ground truth:", result["sample"]["ground_truth"])
    print("Selected experts:", [modalities[i] for i in result["selected"]])
    print("Config used:", result["sample"]["config_dataset"], result["mobe_cfg"])
    print("Class ids:")
    for idx, label in enumerate(classnames):
        print(f"  {idx}: {label}")


if HAS_WIDGETS:
    single_sample = widgets.Dropdown(options=SAMPLE_OPTIONS, value=SELECTED_SAMPLE_ID, description="Sample", layout=widgets.Layout(width="520px"))
    single_route = widgets.Dropdown(options=[("Config route", None), ("MoBE route", True), ("Entropy route", False)], value=None, description="Route")
    single_use_config_lambda = widgets.Checkbox(value=True, description="Use config lambda", indent=False)
    single_lambda = widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="lambda")
    single_gap = widgets.FloatSlider(value=0.07, min=0.0, max=0.50, step=0.01, description="gap_thr")
    single_topk = widgets.IntSlider(value=4, min=1, max=5, step=1, description="topk")
    single_button = widgets.Button(description="Run single-image MoBE", button_style="success", icon="play")
    single_output = widgets.Output()

    def _sync_single_controls(_=None):
        single_lambda.disabled = bool(single_use_config_lambda.value)

    single_use_config_lambda.observe(_sync_single_controls, names="value")
    _sync_single_controls()

    def _single_clicked(_):
        single_button.disabled = True
        try:
            with single_output:
                clear_output(wait=True)
                print("Running BiomedCLIP and MoBE inference...")
            result = run_single_image_comparison(
                sample_id=single_sample.value,
                route_override=single_route.value,
                lambda_override=None if single_use_config_lambda.value else single_lambda.value,
                gap_thr=single_gap.value,
                topk=single_topk.value,
            )
            with single_output:
                clear_output(wait=True)
                plot_single_image_result(result)
        except Exception:
            with single_output:
                clear_output(wait=True)
                traceback.print_exc()
        finally:
            single_button.disabled = False

    single_button.on_click(_single_clicked)
    display(widgets.VBox([
        widgets.HTML("<b>Cell 4: single-image comparison</b>"),
        widgets.HBox([single_sample, single_button]),
        widgets.HBox([single_route, single_use_config_lambda, single_lambda, single_gap, single_topk]),
        single_output,
    ]))
else:
    print("Run result = run_single_image_comparison(); plot_single_image_result(result)")


## Cell 5 — Live One-Dataset BiomedCLIP vs MoBE Comparison

Enter one dataset name, then click **Run comparison**. The left column is BiomedCLIP, and the right column is MoBE. Keep `Use full dataset`, `Use config lambda`, and `Config route` enabled when you want numbers closest to the repository scripts.

In [5]:
def normalize_dataset_name(dataset_name):
    dataset_name = str(dataset_name or DEFAULT_LIVE_DATASET).strip()
    if not dataset_name:
        raise ValueError("Please provide one dataset name, for example chestmnist_224 or breastmnist_224.")
    if "/" in dataset_name:
        raise ValueError("Use one dataset at a time in this live panel. Remove '/' and run a single dataset name.")
    return dataset_name


def prepare_live_target(target):
    target = target.to(device)
    if target.dim() == 0:
        return target.view(1).long()
    if target.dim() == 1:
        return target.long()
    if target.dim() == 2 and target.size(1) == 1:
        return target.squeeze(1).long()
    if target.dim() == 2 and target.size(1) > 1:
        return target.argmax(dim=1).long()
    return target.view(target.size(0), -1).argmax(dim=1).long()


def set_live_status(status_widget, status):
    if status_widget is not None:
        status_widget.value = f"<pre style='white-space: pre-wrap; margin: 0'>{html.escape(status)}</pre>"
    else:
        print(status)


def draw_live_comparison(
    output,
    status_widget,
    xs,
    biomed_acc,
    mobe_acc,
    biomed_conf,
    mobe_conf,
    biomed_rolling_conf,
    mobe_rolling_conf,
    classnames_live,
    biomed_last_probs,
    mobe_last_probs,
    status,
):
    set_live_status(status_widget, status)
    fig, axes = plt.subplots(3, 2, figsize=(15, 10.2), sharex="row")
    fig.suptitle("Live single-dataset inference: BiomedCLIP vs MoBE", fontsize=14, y=1.01)

    columns = [
        ("BiomedCLIP", biomed_acc, biomed_conf, biomed_rolling_conf, biomed_last_probs, "#2f6f9f"),
        ("MoBE", mobe_acc, mobe_conf, mobe_rolling_conf, mobe_last_probs, "#476a2a"),
    ]

    for col, (name, acc, conf, rolling_conf, probs, color) in enumerate(columns):
        x_left = 0.5
        x_right = max(1.5, float(xs[-1]) + 0.5) if len(xs) else 1.5
        acc_arr = np.asarray(acc, dtype=np.float32)
        conf_arr = np.asarray(conf, dtype=np.float32)
        rolling_arr = np.asarray(rolling_conf, dtype=np.float32)

        axes[0, col].plot(xs, acc_arr, color=color, linewidth=2.4, drawstyle="steps-post", alpha=0.9)
        axes[0, col].scatter(xs, acc_arr, color=color, s=28, zorder=4)
        if len(acc_arr):
            axes[0, col].axhline(float(acc_arr[-1]), color=color, linestyle="--", linewidth=1.1, alpha=0.35)
            axes[0, col].text(0.02, 0.92, f"latest {acc_arr[-1]:.1f}%", transform=axes[0, col].transAxes, color=color, weight="bold")
        axes[0, col].set_xlim(x_left, x_right)
        axes[0, col].set_ylim(-5, 105)
        axes[0, col].set_ylabel("accuracy (%)")
        axes[0, col].set_title(f"{name}: cumulative accuracy")
        axes[0, col].grid(alpha=0.25)

        axes[1, col].plot(xs, conf_arr, color=color, alpha=0.42, marker="o", markersize=3, label="per sample")
        axes[1, col].plot(xs, rolling_arr, color=color, linewidth=2.4, drawstyle="steps-post", label="running mean")
        axes[1, col].scatter(xs, rolling_arr, color=color, s=22, zorder=4)
        if len(rolling_arr):
            axes[1, col].axhline(float(rolling_arr[-1]), color=color, linestyle="--", linewidth=1.1, alpha=0.35)
            axes[1, col].text(0.02, 0.88, f"mean {rolling_arr[-1]:.2f}", transform=axes[1, col].transAxes, color=color, weight="bold")
        axes[1, col].set_xlim(x_left, x_right)
        axes[1, col].set_ylim(-0.05, 1.05)
        axes[1, col].set_ylabel("confidence")
        axes[1, col].set_xlabel("samples seen")
        axes[1, col].set_title(f"{name}: confidence")
        axes[1, col].legend(loc="lower right")
        axes[1, col].grid(alpha=0.25)

        axes[2, col].barh(classnames_live, probs, color=color)
        axes[2, col].set_xlim(0, 1)
        axes[2, col].set_xlabel("probability")
        axes[2, col].set_title(f"{name}: current sample prediction")

    plt.tight_layout()
    context = output if output is not None else nullcontext()
    with context:
        clear_output(wait=True)
        display(fig)
    plt.close(fig)


def run_live_dataset_comparison(
    dataset_name=DEFAULT_LIVE_DATASET,
    max_samples=0,
    update_every=5,
    min_refresh_seconds=0.75,
    gap_thr=0.07,
    topk=4,
    route_with_mobe=None,
    lambda_override=None,
    output=None,
    status_widget=None,
):
    from utils import build_test_data_loader, clip_classifier, get_clip_logits

    dataset_name = normalize_dataset_name(dataset_name)
    seed_everything(1)
    context = load_model_context(force=False)
    set_live_status(status_widget, f"Preparing live BiomedCLIP vs MoBE stream for {dataset_name}...")

    mobe_cfg = load_mobe_cfg(dataset_name)
    if route_with_mobe is not None:
        mobe_cfg["route_with_mobe"] = bool(route_with_mobe)
    if lambda_override is not None:
        mobe_cfg["lambda"] = float(lambda_override)

    last_loader_error = None
    for data_root in DATA_ROOTS:
        try:
            loader, classnames_live, templates_live = build_test_data_loader(dataset_name, str(data_root), context["preprocess"])
            data_root_used = data_root
            break
        except FileNotFoundError as exc:
            last_loader_error = exc
    else:
        searched = "\n".join(str(path) for path in DATA_ROOTS)
        raise FileNotFoundError(f"Could not load {dataset_name}. Searched data roots:\n{searched}") from last_loader_error

    clip_weights = clip_classifier(classnames_live, templates_live, context["base_model"], context["tokenizer"]).to(device)
    mu, pi, c1, c2 = mobe_init_from_text_weights(
        clip_weights,
        len(context["expert_models"]),
        init_c1=mobe_cfg["c1"],
        init_c2=mobe_cfg["c2"],
    )

    n_limit = len(loader) if int(max_samples) <= 0 else min(int(max_samples), len(loader))
    update_every = max(1, int(update_every))
    min_refresh_seconds = max(0.0, float(min_refresh_seconds))

    biomed_acc, mobe_acc = [], []
    biomed_conf, mobe_conf = [], []
    biomed_rolling_conf, mobe_rolling_conf = [], []
    biomed_correct_total = 0
    mobe_correct_total = 0
    biomed_last_probs = np.zeros(len(classnames_live), dtype=np.float32)
    mobe_last_probs = np.zeros(len(classnames_live), dtype=np.float32)
    selected_counts = {modality: 0 for modality in context["modalities"]}
    start = time.perf_counter()
    last_draw_time = 0.0

    with torch.no_grad():
        for sample_idx, (images, target) in enumerate(loader, start=1):
            if sample_idx > n_limit:
                break

            target = prepare_live_target(target)
            _, biomed_logits, _, _, _ = get_clip_logits(images, context["base_model"], clip_weights)
            biomed_probs = biomed_logits.softmax(dim=-1)
            biomed_sample_conf, biomed_prediction = biomed_probs.max(dim=-1)
            biomed_correct_total += int(biomed_prediction.eq(target).sum().item())
            biomed_acc.append(100.0 * biomed_correct_total / sample_idx)
            biomed_conf.append(float(biomed_sample_conf.item()))
            biomed_rolling_conf.append(float(np.mean(biomed_conf)))
            biomed_last_probs = biomed_probs.squeeze(0).detach().cpu().numpy()

            expert_feature_list = []
            expert_logit_list = []
            entropies = torch.empty(len(context["expert_models"]), device=device)
            mobe_confidences = torch.empty(len(context["expert_models"]), device=device)

            for expert_idx, expert_model in enumerate(context["expert_models"]):
                features, logits, _, _, _ = get_clip_logits(images, expert_model, clip_weights)
                features = F.normalize(features, dim=-1)
                probs = logits.softmax(dim=-1)
                entropies[expert_idx] = -(probs * (probs + 1e-12).log()).sum(dim=-1).mean()
                _, confidence, _, _ = mobe_forward_and_update_sample(
                    features,
                    expert_idx,
                    mu,
                    pi,
                    c1,
                    c2,
                    thr1=mobe_cfg["thr1"],
                    thr2=mobe_cfg["thr2"],
                    temp=mobe_cfg["temp"],
                )
                expert_feature_list.append(features)
                expert_logit_list.append(logits)
                mobe_confidences[expert_idx] = confidence

            if mobe_cfg["use_mobe"] and mobe_cfg["route_with_mobe"]:
                selected, _, _ = dynamic_k_route(mobe_confidences, lower_is_better=False, topk=min(int(topk), len(context["expert_models"])), gap_thr=gap_thr)
            else:
                selected, _, _ = dynamic_k_route(entropies, lower_is_better=True, topk=min(int(topk), len(context["expert_models"])), gap_thr=gap_thr)

            weights = mixture_weights(selected, entropies, strategy="hard")
            final_logits = torch.zeros_like(expert_logit_list[0])
            for expert_idx in selected:
                selected_counts[context["modalities"][expert_idx]] += 1
                final_logits += weights[expert_idx] * expert_logit_list[expert_idx]

            if mobe_cfg["use_mobe"] and mobe_cfg["lambda"] > 0:
                mobe_mix_probs = 0.0
                for expert_idx in selected:
                    cluster_logits = mobe_cfg["temp"] * (expert_feature_list[expert_idx] @ mu[expert_idx].t())
                    cluster_probs = cluster_logits.softmax(dim=-1)
                    adapted_probs = cluster_probs @ pi[expert_idx]
                    mobe_mix_probs = mobe_mix_probs + weights[expert_idx] * adapted_probs
                final_logits = (1.0 - mobe_cfg["lambda"]) * final_logits + mobe_cfg["lambda"] * torch.log(mobe_mix_probs + 1e-12)

            mobe_probs_final = final_logits.softmax(dim=-1)
            mobe_sample_conf, mobe_prediction = mobe_probs_final.max(dim=-1)
            mobe_correct_total += int(mobe_prediction.eq(target).sum().item())
            mobe_acc.append(100.0 * mobe_correct_total / sample_idx)
            mobe_conf.append(float(mobe_sample_conf.item()))
            mobe_rolling_conf.append(float(np.mean(mobe_conf)))
            mobe_last_probs = mobe_probs_final.squeeze(0).detach().cpu().numpy()

            now = time.perf_counter()
            should_draw = sample_idx == 1 or sample_idx == n_limit or (sample_idx % update_every == 0 and (now - last_draw_time) >= min_refresh_seconds)
            if should_draw:
                elapsed = now - start
                selected_names = [context["modalities"][i] for i in selected]
                status = (
                    f"dataset={dataset_name} | data_root={data_root_used}\n"
                    f"{sample_idx}/{n_limit} samples | target={classnames_live[int(target.item())]} | elapsed={elapsed:.1f}s\n"
                    f"BiomedCLIP: acc={biomed_acc[-1]:.2f}% conf={100.0 * biomed_conf[-1]:.1f}% pred={classnames_live[int(biomed_prediction.item())]}\n"
                    f"MoBE:       acc={mobe_acc[-1]:.2f}% conf={100.0 * mobe_conf[-1]:.1f}% pred={classnames_live[int(mobe_prediction.item())]} | selected={selected_names}"
                )
                draw_live_comparison(
                    output,
                    status_widget,
                    np.arange(1, sample_idx + 1),
                    biomed_acc,
                    mobe_acc,
                    biomed_conf,
                    mobe_conf,
                    biomed_rolling_conf,
                    mobe_rolling_conf,
                    classnames_live,
                    biomed_last_probs,
                    mobe_last_probs,
                    status,
                )
                last_draw_time = now

    return {
        "dataset": dataset_name,
        "biomedclip_accuracy": biomed_acc[-1] if biomed_acc else 0.0,
        "mobe_accuracy": mobe_acc[-1] if mobe_acc else 0.0,
        "biomedclip_mean_confidence": float(np.mean(biomed_conf)) if biomed_conf else 0.0,
        "mobe_mean_confidence": float(np.mean(mobe_conf)) if mobe_conf else 0.0,
        "selected_counts": selected_counts,
        "samples": len(mobe_acc),
    }


if HAS_WIDGETS:
    style = {"description_width": "95px"}
    live_dataset = widgets.Text(value=DEFAULT_LIVE_DATASET, description="Dataset", layout=widgets.Layout(width="360px"), style=style)
    live_all_samples = widgets.Checkbox(value=True, description="Use full dataset", indent=False)
    live_max_samples = widgets.IntSlider(value=40, min=1, max=500, step=1, description="Sample cap", style=style)
    live_update_every = widgets.IntSlider(value=5, min=1, max=50, step=1, description="Refresh", style=style)
    live_min_refresh = widgets.FloatSlider(value=0.75, min=0.0, max=5.0, step=0.25, description="Min sec", style=style)
    live_gap = widgets.FloatSlider(value=0.07, min=0.0, max=0.50, step=0.01, description="gap_thr", style=style)
    live_topk = widgets.IntSlider(value=4, min=1, max=5, step=1, description="topk", style=style)
    live_use_config_lambda = widgets.Checkbox(value=True, description="Use config lambda", indent=False)
    live_lambda = widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="lambda", style=style)
    live_route = widgets.Dropdown(options=[("Config route", None), ("MoBE route", True), ("Entropy route", False)], value=None, description="Route", style=style)
    live_button = widgets.Button(description="Run comparison", button_style="success", icon="play", layout=widgets.Layout(width="180px"))
    live_status = widgets.HTML()
    live_output = widgets.Output()

    def _sync_live(_=None):
        live_max_samples.disabled = bool(live_all_samples.value)
        live_lambda.disabled = bool(live_use_config_lambda.value)

    live_all_samples.observe(_sync_live, names="value")
    live_use_config_lambda.observe(_sync_live, names="value")
    _sync_live()

    def _live_clicked(_):
        live_button.disabled = True
        live_button.description = "Running..."
        try:
            result = run_live_dataset_comparison(
                dataset_name=live_dataset.value,
                max_samples=0 if live_all_samples.value else live_max_samples.value,
                update_every=live_update_every.value,
                min_refresh_seconds=live_min_refresh.value,
                gap_thr=live_gap.value,
                topk=live_topk.value,
                route_with_mobe=live_route.value,
                lambda_override=None if live_use_config_lambda.value else live_lambda.value,
                output=live_output,
                status_widget=live_status,
            )
            live_status.value += f"<pre style='white-space: pre-wrap; margin: 8px 0 0 0'>Final summary: {html.escape(str(result))}</pre>"
        except Exception:
            with live_output:
                clear_output(wait=True)
                traceback.print_exc()
            live_status.value = "<pre style='white-space: pre-wrap; margin: 0'>Check dataset files, configs, and expert checkpoints.</pre>"
        finally:
            live_button.disabled = False
            live_button.description = "Run comparison"

    live_button.on_click(_live_clicked)
    display(widgets.VBox([
        widgets.HTML("<b>Cell 5: live one-dataset comparison</b>"),
        widgets.HBox([live_dataset, live_all_samples, live_max_samples, live_button]),
        widgets.HBox([live_update_every, live_min_refresh, live_topk]),
        widgets.HBox([live_gap, live_use_config_lambda, live_lambda, live_route]),
        live_status,
        live_output,
    ]))
else:
    print("Run run_live_dataset_comparison(dataset_name='chestmnist_224', max_samples=0) when widgets are unavailable.")


## Cell 6 — Takeaways

- The sample-image path uses BiomedCLIP and modality expert checkpoints.
- MoBE selects experts with dynamic routing and can blend expert logits with adapted Bayesian probabilities.
- The live panel is the closest notebook analogue to running the repository scripts, while still showing accuracy/confidence as inference progresses.
- To match script-style runs, use one dataset at a time, keep full-dataset mode enabled, and keep config route/lambda enabled.
